In [ ]:
# %% Deep learning - Section 24.222
#    Lorem ipsum

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2
import torchvision
import torchvision.transforms as T
import torch.nn.utils         as utils
import random

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from matplotlib.gridspec              import GridSpec
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
# %% Generate some text

# generated at https://www.lipsum.com/
text = 'Lorem ipsum dolor sit amet, consectetur adipiscing elit. Praesent molestie sapien auctor eleifend egestas. Fusce at purus sodales, viverra nunc quis, consequat augue. Vestibulum eget tempus lorem, et blandit dui. Suspendisse ac gravida odio. Maecenas consequat tristique mi, vitae rutrum lacus pulvinar vitae. Nunc ullamcorper nulla eu velit vehicula, vitae facilisis erat dignissim. Proin consectetur nec lacus ac pellentesque. Nulla purus ligula, commodo id tellus id, efficitur varius massa. Phasellus et volutpat felis, gravida imperdiet justo. Cras metus velit, aliquet et tristique sit amet, elementum ultrices dui. Nullam condimentum quis orci quis pretium. Mauris tincidunt ante nec ex tristique, a commodo quam eleifend. Nam convallis ultrices magna fringilla porta. Phasellus non lobortis nisi. Donec nec lectus ligula. Maecenas id purus at lectus auctor finibus sit amet et enim. Vivamus nibh urna, dapibus sed porta in, sodales vitae elit. Fusce sed facilisis elit, ut porta massa. Vivamus blandit congue erat eget rutrum. Nullam mollis, eros et laoreet euismod, nunc mi condimentum eros, mollis pretium mi orci in nibh. Pellentesque rhoncus justo et pretium tempor. Ut gravida egestas quam, sit amet sagittis tortor scelerisque in. Vestibulum sed odio urna. Donec semper quis erat quis laoreet. Ut malesuada volutpat sem ac luctus. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia curae; Praesent sed bibendum sapien, id imperdiet elit. Vestibulum erat lorem, finibus eu enim non, posuere tempus velit. Vestibulum a massa id orci interdum malesuada eu vel tellus. Proin tempus viverra scelerisque. Nullam suscipit laoreet nisl, id consequat sem porttitor et. Integer congue urna lacus, ac feugiat arcu tincidunt eget. Aliquam erat volutpat. Vivamus accumsan semper gravida. Mauris porta magna vitae semper hendrerit. Vestibulum urna nunc, faucibus sit amet auctor sed, scelerisque nec est. Nulla ut sagittis urna. Proin fermentum turpis non iaculis tincidunt. Maecenas scelerisque rutrum hendrerit. Sed fermentum vehicula molestie. Sed nec rutrum nisi. Class aptent taciti sociosqu ad litora torquent per conubia nostra, per inceptos himenaeos. Cras malesuada, magna in ornare pretium, leo tellus sodales tortor, sit amet fermentum nunc odio eu enim. Quisque placerat eros ornare nulla vulputate, at efficitur sem convallis. Sed libero risus, viverra a turpis a, sollicitudin feugiat neque. Fusce vitae erat commodo, consectetur lacus vel, sollicitudin lorem. Nunc sed risus arcu. Pellentesque nec eleifend risus, a fringilla odio. Sed auctor augue a rutrum maximus. Maecenas suscipit tellus sem, vitae suscipit nisl euismod a. Phasellus elementum sodales urna, ac fringilla mi malesuada id. Suspendisse sollicitudin rhoncus dolor ut consequat. Duis tincidunt quis neque nec tincidunt. Fusce vitae sagittis nulla. Suspendisse ac varius mauris. Maecenas dapibus posuere velit, nec pellentesque quam sagittis a. Nunc aliquet justo vitae justo pharetra consectetur. Nam porttitor at nisl sit amet ullamcorper. Sed rutrum, nulla ac porttitor pulvinar, nisi leo hendrerit magna, non luctus nibh risus eget est. Quisque pulvinar rutrum vehicula. Ut tempor placerat sollicitudin. Etiam pharetra sit amet nulla at fringilla. Pellentesque feugiat odio ligula, ac ullamcorper leo vulputate a. Vestibulum placerat interdum arcu, sit amet ullamcorper ipsum finibus sed. Aliquam erat volutpat. Nam tincidunt, augue eu eleifend dictum, tellus sem blandit sem, et pulvinar ex purus sed leo. Nullam ultricies tincidunt sem, imperdiet condimentum ex porttitor at. Nunc id lacus sit amet nibh elementum dignissim. Nam facilisis tincidunt tincidunt. Suspendisse in mauris vel dui imperdiet facilisis. Aenean eu neque tortor. Cras sit amet mi nibh. Mauris sit amet feugiat nulla. Nam ac leo ipsum. Vestibulum id enim sit amet est pharetra consectetur. Vestibulum et lacus sed ipsum placerat blandit vitae quis nisl. Curabitur lacus est, euismod non accumsan sed, accumsan nec lectus. Pellentesque habitant morbi tristique senectus et netus et malesuada fames ac turpis egestas. Maecenas ultrices eros in erat molestie interdum. Nunc et tellus orci. Maecenas et magna ornare mauris sodales malesuada. Duis iaculis ipsum non laoreet porta. Aenean vitae purus tempor, porttitor arcu id, bibendum enim. Aliquam faucibus congue eros, eget feugiat risus venenatis a. Duis malesuada, sem eu mattis placerat, velit lectus varius tellus, eget placerat nibh quam non turpis. Donec auctor pellentesque odio, nec pulvinar nisi fermentum eget. Mauris eget eleifend metus. Mauris venenatis arcu semper erat facilisis, malesuada viverra tortor imperdiet. Nunc ut quam sit amet ex varius euismod. Mauris eleifend lectus venenatis risus mattis consequat. Nulla a eros non erat egestas consequat nec volutpat neque. In diam nulla, mollis ut semper nec, vulputate luctus odio. Morbi ac elementum quam, ut vestibulum sem. Ut tincidunt sapien ac fermentum ullamcorper. Cras convallis tortor quis malesuada dignissim. Suspendisse rutrum cursus diam, in consequat nisi vulputate sit amet. Nunc euismod consectetur libero eu pulvinar. Ut finibus scelerisque lectus vel auctor. Vivamus congue non sem et tincidunt. Vestibulum vehicula erat sed nisi mattis aliquet. Class aptent taciti sociosqu ad litora torquent per conubia nostra, per inceptos himenaeos. Etiam pulvinar tortor enim, vel blandit mauris sodales quis. Ut bibendum dui non posuere pellentesque. Phasellus metus diam, blandit accumsan porta a, pharetra nec nulla. Nam pulvinar, lacus et ornare luctus, magna orci tincidunt lorem, porttitor tincidunt enim mi a ligula. Pellentesque habitant morbi tristique senectus et netus et malesuada fames ac turpis egestas. Etiam quis mi porta, mattis velit vel, rhoncus nisi. Etiam lobortis placerat lacus.'.lower()
print(text)
print(len(text))


In [ ]:
# %% Get all the unique characters

unique_characters = set(text)

print(unique_characters)
print(len(unique_characters))


In [ ]:
# %% look-up tables to convert characters to indices and vice-versa

# Dictionaties are just ideal (two implementation but same concept)
number2letter = dict(enumerate(unique_characters))
letter2number = { l:i for i,l in number2letter.items() }

print(letter2number['e'])
number2letter


In [ ]:
# %% Convert text from characters into numbers

# note the inputs to zeros()
data = torch.zeros( (len(text),1), dtype=torch.int64, device=device)
for i,ch in enumerate(text):
    data[i] = letter2number[ch]

print(data)

# Plot
phi = (1 + np.sqrt(5))/2
plt.figure(figsize=(phi*6,6))

plt.plot(data.cpu().numpy(),'k.')
plt.xlabel('Character index')
plt.ylabel('Character numerical label')
plt.title('Character sequence, or how a model sees strings of letters')

plt.savefig('figure55_lorem_ipsum.png')
plt.show()
files.download('figure55_lorem_ipsum.png')


In [87]:
# %% LSTM class

class LSTM(nn.Module):
    def __init__(self,input_size,output_size,hidden_size,num_layers):
        super().__init__()

        # Embedding layer (just a wrapper for nn.Linear to deal with the
        # matrices mapping)
        self.embedding = nn.Embedding(input_size,input_size)


        # LSTM layer
        self.lstm = nn.LSTM(input_size,hidden_size,num_layers)

        # Linear output layer (output size is the same as input size, that is,
        # the number of unique characters in the text, then when generating the
        # new text we select the letter with highest probability)
        self.out = nn.Linear(hidden_size,output_size)

    def forward(self,x,h):

        # Embedding layer pass
        embedding = self.embedding(x)

        # Run through the LSTM layer
        y,h = self.lstm(embedding,h)

        # Output (linear) layer and numerical values for h (i.e., the three
        # inputs, input proper, hidden state, cells state)
        y = self.out(y)
        h = (h[0].detach(),h[1].detach())

        return y,h


In [88]:
# %% Metaparameters, model instance, loss function and optimizer

# Metaparameters
hidden_size = 512   # size of hidden state (units in hidden layers)
seq_length  = 80    # length of sequence (num of characters)
num_layers  = 3     # number of stacked hidden layers
epochs      = 10    # training epochs

# Model instance
lstm = LSTM(len(unique_characters),len(unique_characters),hidden_size,num_layers).to(device)

# Loss function and optimizer (categorical problem so CE loss)
loss_fun  = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm.parameters(),lr=.001)


In [ ]:
# %% Plotting

# Get the randomly initialized embeddings matrix
I = next(lstm.embedding.named_parameters())

phi = (1 + np.sqrt(5))/2
plt.figure(figsize=(phi*6,6))

plt.imshow(I[1].cpu().detach(),cmap='plasma')
plt.title('Randomly initialized embeddings matrix')

plt.savefig('figure56_lorem_ipsum.png')
plt.show()
files.download('figure56_lorem_ipsum.png')


In [90]:
# %% Train the model (takes ~13 mins on GPU)

# Preallocate losses and store initial embeddings
losses   = np.zeros(epochs)
I_before = lstm.embedding.weight.detach().cpu().clone()

# Loop over epochs
for epochi in range(epochs):

    # Initialize loss and hidden state for this epoch
    txt_loss = 0
    hidden_state = None

    # Loop through the entire text character-wise
    for txt_loc in range(0,len(text)-seq_length):

        # Get input and target (target is just a +1 shifted version of input, so
        # the model is trained to learn the next character)
        x = data[txt_loc   : txt_loc+seq_length  ]
        y = data[txt_loc+1 : txt_loc+seq_length+1]

        # Forward propagation
        output,hidden_state = lstm(x,None)

        # Loss
        loss      = loss_fun(torch.squeeze(output),torch.squeeze(y))
        txt_loss += loss.item()

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Average losses for this epoch (run through the entire text)
    losses[epochi] = txt_loss / txt_loc

# Store trained embeddings
I_after = lstm.embedding.weight.detach().cpu()


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5))/2
plt.figure(figsize=(phi*6,6))

plt.plot(losses,'s-')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss')

plt.savefig('figure57_lorem_ipsum.png')
plt.show()
files.download('figure57_lorem_ipsum.png')


In [ ]:
# %% Reconstruct a character sequence from number sequence

# Here x is the last snippet of trainig data
t = ''
for l in x:
  t += number2letter[l.item()]

print(t)


In [ ]:
# %% Generate new lorem ipsum text

# Number of characters to generate
lorem_length = 300

# Random character from data to begin with (x here is a letter)
x     = torch.tensor(letter2number['x']).view(1,1).to(device)
lorem = number2letter[x.item()]

# Initialize the hidden state
hidden_state = None

# Generate text
for i in range(lorem_length):

    # Push a letter though the LSTM
    output,hidden_state = lstm(x,hidden_state)

    # Get the maximum output and replace input data (this is deterministic
    # solution; another way would be use softmax and pick the output in a more
    # probabilistic way)
    index   = torch.argmax(output).item()
    x[0][0] = index

    # Append current output to the text after converting to character
    lorem += number2letter[index]

# What does it say?
print(lorem)


In [ ]:
# %% Plotting

# Show hidden states
phi = (1 + np.sqrt(5))/2
plt.figure(figsize=(phi*6,6))

for i in range(num_layers):
    plt.plot(hidden_state[0][i,0,:].cpu().numpy(),'o',label='Layer {}'.format(i))
    plt.xlabel('Character index')
    plt.ylabel('Hidden state')

plt.legend()
plt.title('Hidden states')

plt.savefig('figure58_lorem_ipsum.png')
plt.show()
files.download('figure58_lorem_ipsum.png')


In [ ]:
# %% Plotting

# Get the learnt embeddings matrix
I = next(lstm.embedding.named_parameters())
I = I[1].cpu().detach().numpy()

phi = (1 + np.sqrt(5))/2
plt.figure(figsize=(phi*6,6))

plt.imshow(I,cmap='plasma')
plt.title('Trained embeddings matrix')

plt.savefig('figure59_lorem_ipsum.png')
plt.show()
files.download('figure59_lorem_ipsum.png')


In [ ]:
# %% Plotting

# Difference pre-post training
diff = (I_after - I_before).numpy()

phi = (1 + np.sqrt(5))/2
plt.figure(figsize=(phi*6,6))

plt.imshow(diff, cmap='plasma')
plt.title('Embedding changes')
plt.colorbar()

plt.savefig('figure60_lorem_ipsum.png')
plt.show()
files.download('figure60_lorem_ipsum.png')


In [ ]:
# %% Investigate embedding matrix

# For example compute PCA
d,V = np.linalg.eig(I@I.T)

# Plot
phi = (1 + np.sqrt(5))/2
fig,axs = plt.subplots(1,2,figsize=(phi*6,6))

axs[0].imshow(np.corrcoef(I),vmin=-.5,vmax=.5,cmap='plasma')
axs[0].set_title('Correlation of embeddings')
axs[0].set_xticks(range(len(letter2number.keys())))
axs[0].set_xticklabels(letter2number.keys())
axs[0].set_yticks(range(len(letter2number.keys())))
axs[0].set_yticklabels(letter2number.keys())

axs[1].plot(d,'s-')
axs[1].set_xlabel('Component')
axs[1].set_ylabel('Eigenvalue')
axs[1].set_title('Eigenspectrum of embeddings')
axs[1].set_aspect('auto')

plt.tight_layout()

plt.savefig('figure61_lorem_ipsum.png')
plt.show()
files.download('figure61_lorem_ipsum.png')


In [ ]:
# %% Exercise 1
#    The model seems to work well with the current parameters. But does it need so many hidden units and layers?
#    Try it again using one layer and 50 hidden units. Compare the loss function (quantitative) and the generated
#    text (qualitative) with the current instantiation.

# Well, the output is quite worse, but still quite a surprising pseudo-text:
#
#    x portor. etiam pursco erat egestas. etiam quis portitor tincidunt enim mi
#    a ligula. curis egestas. etiam quis portitor tincidunt enim mi a ligula.
#    curis egestas. etiam quis portitor tincidunt enim mi


In [ ]:
# %% Exercise 2
#    In the video, I discussed that the embeddings matrix does not need to be square. Make it 2x as wide (thus, it
#    will be a 26x52 matrix). What needs to be changed in the code?

# Seems like there is some sort of improvement. New output:
#
#    x taciti sociosqu aptent taciti sociosqu ad litora torquent per conubia
#    nostra, per inceptos himenaeos. etiam pulvinar, lacus et netus et malesuada
#    fames ac turpis egestas. etiam quis mi porta, mattis

# Change the model with:
self.embedding = nn.Embedding(input_size,2*input_size)
self.lstm = nn.LSTM(2*input_size,hidden_size,num_layers)


In [ ]:
# %% Exercise 3
#    Lorem ipsum is already gibberish text (though each word is a plausible word). Replace the text with real text,
#    e.g., something you wrote, or perhaps copy text from a wiki page or a short story. Is the generated text still
#    nonsense words or real words? (Note that training could take a really long time if you use a really long text.)

# Trying with the incpit of the Peloponnesian war. Here's the output:
#
#     xanian preates for the pirates used to plunder one another, and indeed all coast populations, whether seagh doald fandans of aid
#                    for the pirates used to plunder one another, and indeed all coast populations, whether seagh doald fandans of aid
#                    for the pirates used to plunder one another, and indeed al

# Alternative text (takes ~ mins on GPU)
text = 'Thucydides, an Athenian, wrote the history of the war between the Peloponnesians and the Athenians, beginning at the moment that it broke out, and believing that it would be a great war, and more worthy of relation than any that had preceded it. This belief was not without its grounds. The preparations of both the combatants were in every department in the last state of perfection; and he could see the rest of the Hellenic race taking sides in the quarrel; those who delayed doing so at once having it in contemplation. [2] Indeed this was the greatest movement yet known in history, not only of the Hellenes, but of a large part of the barbarian world—I had almost said of mankind. [3] For though the events of remote antiquity, and even those that more immediately precede the war, could not from lapse of time be clearly ascertained, yet the evidences which an inquiry carried as far back as was practicable leads me to trust, all point to the conclusion that there was nothing on a great scale, either in war or in other matters. For instance, it is evident that the country now called Hellas had in ancient times no settled population; on the contrary, migrations were of frequent occurrence, the several tribes readily abandoning their homes under the pressure of superior numbers. [2] Without commerce, without freedom of communication either by land or sea, cultivating no more of their territory than the exigencies of life required, destitute of capital, never planting their land （for they could not tell when an invader might not come and take it all away, and when he did come they had no walls to stop him）, thinking that the necessities of daily sustenance could be supplied at one place as well as another, they cared little for shifting their habitation, and consequently neither built large cities nor attained to any other form of greatness. [3] The richest soils were always most subject to this change of masters; such as the district now called Thessaly, Boeotia, most of the Peloponnese, Arcadia excepted, and the most fertile parts of the rest of Hellas. [4] The goodness of the land favoured the aggrandizement of particular individuals, and thus created faction which proved a fertile source of ruin. It also invited invasion. [5] Accordingly Attica, from the poverty of its soil enjoying from a very remote period freedom from faction, never changed its inhabitants. [6] And here is no inconsiderable exemplification of my assertion, that the migrations were the cause of there being no correspondent growth in other parts. The most powerful victims of war or faction from the rest of Hellas took refuge with the Athenians as a safe retreat; and at an early period, becoming naturalized, swelled the already large population of the city to such a height that Attica became at last too small to hold them, and they had to send out colonies to Ionia. There is also another circumstance that contributes not a little to my conviction of the weakness of ancient times. Before the Trojan war there is no indication of any common action in Hellas, [2] nor indeed of the universal prevalence of the name; on the contrary, before the time of Hellen, son of Deucalion, no such appellation existed, but the country went by the names of the different tribes, in particular of the Pelasgian. It was not till Hellen and his sons grew strong in Phthiotis, and were invited as allies into the other cities, that one by one they gradually acquired from the connection the name of Hellenes; though a long time elapsed before that name could fasten itself upon all. [3] The best proof of this is furnished by Homer. Born long after the Trojan war, he nowhere calls all of them by that name, nor indeed any of them except the followers of Achilles from Phthiotis, who were the original Hellenes: in his poems they are called Danaans, Argives, and Achaeans. He does not even use the term barbarian, probably because the Hellenes had not yet been marked off from the rest of the world by one distinctive appellation. [4] It appears therefore that the several Hellenic communities, comprising not only those who first acquired the name, city by city, as they came to understand each other, but also those who assumed it afterwards as the name of the whole people, were before the Trojan war prevented by their want of strength and the absence of mutual intercourse from displaying any collective action. Indeed, they could not unite for this expedition till they had gained increased familiarity with the sea. And the first person known to us by tradition as having established a navy is Minos. He made himself master of what is now called the Hellenic sea, and ruled over the Cyclades, into most of which he sent the first colonies, expelling the Carians and appointing his own sons governors; and thus did his best to put down piracy in those waters, a necessary step to secure the revenues for his own use. For in early times the Hellenes and the barbarians of the coast and islands, as communication by sea became more common, were tempted to turn pirates, under the conduct of their most powerful men; the motives being to serve their own cupidity and to support the needy. They would fall upon a town unprotected by walls, and consisting of a mere collection of villages, and would plunder it; indeed, this came to be the main source of their livelihood, no disgrace being yet attached to such an achievement, but even some glory. [2] An illustration of this is furnished by the honor with which some of the inhabitants of the continent still regard a successful marauder, and by the question we find the old poets everywhere representing the people as asking of voyagers ‘Are they pirates?’ - as if those who are asked the question would have no idea of disclaiming the imputation, or their interrogators of reproaching them for it. [3] The same rapine prevailed also by land. And even at the present day many parts of Hellas still follow the old fashion, the Ozolian Locrians, for instance, the Aetolians, the Acarnanians, and that region of the continent; and the custom of carrying arms is still kept up among these continentals, from the old piratical habits. The whole of Hellas used once to carry arms, their habitations being unprotected, and their communication with each other unsafe; indeed, to wear arms was as much a part of everyday life with them as with the barbarians. [2] And the fact that the people in these parts of Hellas are still living in the old way points to a time when the same mode of life was once equally common to all. [3] The Athenians were the first to lay aside their weapons, and to adopt an easier and more luxurious mode of life; indeed, it is only lately that their rich old men left off the luxury of wearing undergarments of linen, and fastening a knot of their hair with a tie of golden grasshoppers, a fashion which spread to their Ionian kindred, and long prevailed among the old men there. [4] On the contrary a modest style of dressing, more in conformity with modern ideas, was first adopted by the Lacedaemonians, the rich doing their best to assimilate their way of life to that of the common people. [5] They also set the example of contending naked, publicly stripping and anointing themselves with oil in their gymnastic exercises. Formerly, even in the Olympic contests, the athletes who contended wore belts across their middles; and it is but a few years since that the practice ceased. To this day among some of the barbarians, especially in Asia, when prizes for boxing and wrestling are offered, belts are worn by the combatants. [6] And there are many other points in which a likeness might be shown between the life of the Hellenic world of old and the barbarian of to-day. With respect to their towns, later on, at an era of increased facilities of navigation and a greater supply of capital, we find the shores becoming the site of walled towns, and the isthmuses being occupied for the purposes of commerce, and defence against a neighbor. But the old towns, on account of the great prevalence of piracy, were built away from the sea, whether on the islands or the continent, and still remain in their old sites. For the pirates used to plunder one another, and indeed all coast populations, whether seafaring or not.'.lower()
print(text)
print(len(text))
